# Practical 12: Basic AI Learning Models: k-NN, Decision Tree, Naive Bayes and Perceptron

## Objective
Implement simple learning algorithms and compare their predictions on a small toy dataset.

**Course:** Artificial Intelligence (01AI0511)  
**Language:** Python 3

> Run the notebook from top to bottom. The examples use small, readable datasets so the algorithms can be understood and modified.

## 1. Prepare a toy dataset
Each sample has two numeric features and a binary class label.

In [ ]:
X = [
    [1.0, 1.0], [1.5, 1.2], [2.0, 1.0], [2.0, 2.0],
    [4.0, 4.0], [4.5, 4.2], [5.0, 4.0], [5.0, 5.0]
]
y = ["A", "A", "A", "A", "B", "B", "B", "B"]

def train_test_split(X, y, test_size=2):
    return X[:-test_size], X[-test_size:], y[:-test_size], y[-test_size:]

X_train, X_test, y_train, y_test = train_test_split(X, y)
print("Training samples:", len(X_train), "| Test samples:", len(X_test))

## 2. k-Nearest Neighbors

In [ ]:
from math import sqrt
from collections import Counter

def euclidean(a, b):
    return sqrt(sum((x-y)**2 for x, y in zip(a, b)))

def knn_predict(X_train, y_train, sample, k=3):
    distances = sorted(
        (euclidean(sample, point), label)
        for point, label in zip(X_train, y_train)
    )
    nearest_labels = [label for _, label in distances[:k]]
    return Counter(nearest_labels).most_common(1)[0][0]

knn_predictions = [knn_predict(X_train, y_train, x, k=3) for x in X_test]
print("k-NN predictions:", knn_predictions)

## 3. Decision tree (small categorical demonstration)
A one-level decision tree chooses the feature and threshold with the best classification accuracy.

In [ ]:
def majority_label(labels):
    return Counter(labels).most_common(1)[0][0]

def train_stump(X, y):
    best = None
    for feature in range(len(X[0])):
        thresholds = sorted(set(row[feature] for row in X))
        for threshold in thresholds:
            left = [label for row, label in zip(X, y) if row[feature] <= threshold]
            right = [label for row, label in zip(X, y) if row[feature] > threshold]
            if not left or not right:
                continue
            left_label, right_label = majority_label(left), majority_label(right)
            predictions = [
                left_label if row[feature] <= threshold else right_label for row in X
            ]
            accuracy = sum(a == b for a, b in zip(predictions, y)) / len(y)
            if best is None or accuracy > best["accuracy"]:
                best = {"feature": feature, "threshold": threshold,
                        "left": left_label, "right": right_label,
                        "accuracy": accuracy}
    return best

def stump_predict(model, sample):
    return model["left"] if sample[model["feature"]] <= model["threshold"] else model["right"]

tree = train_stump(X_train, y_train)
tree_predictions = [stump_predict(tree, x) for x in X_test]
print("Decision stump:", tree)
print("Predictions:", tree_predictions)

## 4. Naive Bayes (Gaussian)

In [ ]:
def fit_gaussian_nb(X, y):
    model = {}
    for label in sorted(set(y)):
        rows = [row for row, target in zip(X, y) if target == label]
        means = [sum(col)/len(col) for col in zip(*rows)]
        variances = [
            sum((row[j]-means[j])**2 for row in rows)/len(rows) + 1e-9
            for j in range(len(means))
        ]
        model[label] = {"prior": len(rows)/len(y), "means": means, "variances": variances}
    return model

def gaussian_nb_predict(model, sample):
    log_scores = {}
    for label, params in model.items():
        score = __import__("math").log(params["prior"])
        for x, mean, var in zip(sample, params["means"], params["variances"]):
            score += -0.5 * __import__("math").log(2 * __import__("math").pi * var)
            score += -((x-mean)**2)/(2*var)
        log_scores[label] = score
    return max(log_scores, key=log_scores.get)

nb_model = fit_gaussian_nb(X_train, y_train)
nb_predictions = [gaussian_nb_predict(nb_model, x) for x in X_test]
print("Gaussian Naive Bayes predictions:", nb_predictions)

## 5. Perceptron
A simple binary linear classifier. Labels A and B are mapped to -1 and +1.

In [ ]:
def train_perceptron(X, y, learning_rate=0.1, epochs=20):
    weights = [0.0] * len(X[0])
    bias = 0.0
    targets = [1 if label == "B" else -1 for label in y]
    for _ in range(epochs):
        for row, target in zip(X, targets):
            activation = sum(w*x for w, x in zip(weights, row)) + bias
            prediction = 1 if activation >= 0 else -1
            if prediction != target:
                for j in range(len(weights)):
                    weights[j] += learning_rate * target * row[j]
                bias += learning_rate * target
    return weights, bias

def perceptron_predict(model, sample):
    weights, bias = model
    score = sum(w*x for w, x in zip(weights, sample)) + bias
    return "B" if score >= 0 else "A"

perceptron = train_perceptron(X_train, y_train)
perceptron_predictions = [perceptron_predict(perceptron, x) for x in X_test]
print("Perceptron predictions:", perceptron_predictions)

## 6. Compare models

In [ ]:
def accuracy(actual, predicted):
    return sum(a == p for a, p in zip(actual, predicted)) / len(actual)

results = {
    "k-NN": knn_predictions,
    "Decision stump": tree_predictions,
    "Gaussian Naive Bayes": nb_predictions,
    "Perceptron": perceptron_predictions
}
for name, predictions in results.items():
    print(f"{name:22s}: {accuracy(y_test, predictions):.2%}")
print("Actual labels:", y_test)

## Conclusion
The algorithms use different assumptions: k-NN uses nearby examples, a decision tree uses feature splits, Naive Bayes uses conditional probability, and the perceptron learns a linear decision boundary. Results on this tiny dataset are only illustrative.

## Result
The practical has been implemented. Review the outputs, experiment with the inputs, and add your observations before submission.